# 01 — First inference: CPU, then GPU

**Question:** how does running the same open-weight model on a GPU change generation time?

We will run `Qwen/Qwen2.5-1.5B-Instruct`, inspect its inputs and response, and save a small baseline. No training or serving framework is involved.

Run cells in order. Start in Colab with a CPU runtime, then download your result before switching to a GPU runtime and rerunning from the top. Locally, use a Jupyter Python kernel. The first run downloads several GB of weights; FP32 model parameters occupy roughly 6 GB of RAM, with additional memory needed for loading and execution.

**Predict first:** will the first request take longer than later requests? Why might generating 64 tokens take longer than processing a 64-token prompt?

## 1. Install dependencies

Keep the runtime's compatible PyTorch installation when it satisfies the requirement. Transformers is pinned for this lesson; exact installed versions are saved with every run. If packages were already imported before installation, restart the kernel/runtime and continue from the next cell.

In [1]:
# Install the inference libraries in this notebook's Python environment.
%pip install -q "torch>=2.6,<3" "transformers==4.57.1" psutil

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Select hardware and set the experiment

`FORCE_CPU = False` automatically selects CUDA, then Apple MPS, then CPU. Set it to `True` to benchmark CPU even when a GPU is present. Availability means usable by this PyTorch installation; a physically installed GPU may still need a compatible runtime.

Keep the prompt, model revision, precision, token budget, and repeat count unchanged between the CPU and GPU runs. FP32 is deliberate on both devices; lower precision is a later experiment. Changing these settings requires rerunning the following cells, preferably in a fresh runtime.

In [3]:
import csv  # Export each response and its measurements as a table.
import json  # Save complete benchmark results in a structured file.
import os  # Read the number of logical CPU cores.
import platform  # Identify the operating system, CPU, and Python version.
import statistics  # Summarize repeated measurements with the median.
import time  # Measure elapsed wall-clock time.
import uuid  # Keep filenames unique even when runs finish in the same second.
from datetime import datetime, timezone  # Label result files with a UTC timestamp.
from importlib.metadata import version  # Record the installed library versions.
from pathlib import Path  # Build and create output paths.

import psutil  # Read the system's physical RAM capacity.
import torch  # Provide tensors, device selection, and inference execution.
from huggingface_hub import model_info  # Resolve a model revision to an exact commit.
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig  # Load the model/tokenizer and configure generation.

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Select the pretrained instruction model.
MODEL_REVISION = "main"  # For the GPU run, paste the resolved revision from the CPU result.
FORCE_CPU = False  # Set True for the CPU baseline even when a GPU is available.
MAX_NEW_TOKENS = 64  # Cap the response length; generation may stop earlier.
MEASURED_RUNS = 3  # Repeat the same request after one warmup.
PROMPT = "Explain the difference between a CPU and a GPU in three short sentences."  # Keep the workload fixed.
DTYPE = torch.float32  # Use FP32 on both devices for the first comparison.

torch.manual_seed(42)  # Fix the random seed; hardware can still affect numerical results.
torch.set_float32_matmul_precision("highest")  # Avoid reduced-precision FP32 matrix multiplication.
cuda_available = torch.cuda.is_available()  # Check whether PyTorch can use a CUDA GPU.
mps_available = torch.backends.mps.is_available()  # Check whether PyTorch can use an Apple GPU.
device = torch.device("cpu" if FORCE_CPU else "cuda:0" if cuda_available else "mps" if mps_available else "cpu")  # Prefer a GPU unless CPU is forced.

def synchronize():
    # GPU work is asynchronous; wait for it before reading the wall-clock timer.
    if device.type == "cuda":
        torch.cuda.synchronize(device)  # Wait for queued CUDA work to finish.
    elif device.type == "mps":
        torch.mps.synchronize()  # Wait for queued Apple GPU work to finish.

# Save the hardware and software context needed to interpret this benchmark.
environment = {
    "python": platform.python_version(),  # Python interpreter version.
    "packages": {name: version(name) for name in ["torch", "transformers", "huggingface-hub", "tokenizers", "safetensors", "psutil"]},  # Installed dependency versions.
    "platform": platform.platform(),  # Operating system and architecture.
    "cpu": platform.processor() or platform.machine(),  # Available CPU identification.
    "logical_cpu_count": os.cpu_count(),  # Logical cores exposed to this process.
    "torch_cpu_threads": torch.get_num_threads(),  # Threads PyTorch uses for CPU operations.
    "system_ram_gib": psutil.virtual_memory().total / 2**30,  # Physical system RAM in GiB.
    "cuda_available": cuda_available,  # Whether CUDA was detected, even if CPU is forced.
    "mps_available": mps_available,  # Whether Apple GPU support was detected.
    "device": str(device),  # Device actually selected for inference.
    "force_cpu": FORCE_CPU,  # Record whether automatic selection was overridden.
    "dtype": str(DTYPE),  # Weight precision used for this comparison.
    "cuda_build": torch.version.cuda,  # CUDA version associated with the PyTorch build.
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else "Apple MPS" if device.type == "mps" else None,  # Selected GPU, if any.
    "gpu_total_gib": torch.cuda.get_device_properties(device).total_memory / 2**30 if device.type == "cuda" else None,  # CUDA device capacity; unavailable elsewhere.
}
print(json.dumps(environment, indent=2))  # Inspect the selected environment before loading weights.

{
  "python": "3.13.18",
  "packages": {
    "torch": "2.14.1+cpu",
    "transformers": "4.57.1",
    "huggingface-hub": "0.36.2",
    "tokenizers": "0.22.2",
    "safetensors": "0.8.0",
    "psutil": "5.9.5"
  },
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "cpu": "x86_64",
  "logical_cpu_count": 8,
  "torch_cpu_threads": 4,
  "system_ram_gib": 50.99027633666992,
  "cuda_available": false,
  "mps_available": false,
  "device": "cpu",
  "force_cpu": false,
  "dtype": "torch.float32",
  "cuda_build": null,
  "gpu_name": null,
  "gpu_total_gib": null
}


## 3. Load the tokenizer and model

The tokenizer maps text to token IDs. The model loads pretrained weights and predicts tokens. We resolve one model revision and use it for both downloads.

The timer below includes revision lookup, any downloads, loading, and moving weights to the selected device. Cached downloads make this time vary; it is kept separate from generation latency. `eval()` disables training behavior such as dropout. No automatic CPU offloading is used: the whole model must fit on the selected device.

In [4]:
load_start = time.perf_counter()  # Start timing revision lookup, downloads, and model loading.
resolved_revision = model_info(MODEL_NAME, revision=MODEL_REVISION).sha  # Resolve one exact model commit.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=resolved_revision)  # Load the matching text-to-token rules.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,  # Load the chosen model's architecture and pretrained weights.
    revision=resolved_revision,  # Use the same commit as the tokenizer.
    dtype=DTYPE,  # Load weights in the explicitly selected precision.
    attn_implementation="sdpa",  # Use PyTorch's scaled dot-product attention implementation.
).to(device)  # Move the entire model to the selected CPU or GPU.
model.eval()  # Disable training-specific behavior such as dropout.
synchronize()  # Include completed device transfers in the loading measurement.
download_and_load_seconds = time.perf_counter() - load_start  # Keep loading separate from generation timing.

print("Resolved revision:", resolved_revision)  # Copy this revision into the comparison run.
print("Parameter device:", next(model.parameters()).device)  # Confirm where model weights live.
print("Parameter dtype:", next(model.parameters()).dtype)  # Confirm the actual weight precision.
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")  # Count the model's parameters.
print(f"Download + load + device transfer: {download_and_load_seconds:.2f} seconds")  # Display setup time.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Resolved revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
Parameter device: cpu
Parameter dtype: torch.float32
Parameters: 1,543,714,304
Download + load + device transfer: 34.77 seconds


## 4. Inspect the model inputs

An instruction model expects a chat template that marks message roles and where the assistant response begins. The template itself adds input tokens.

`input_ids` has shape `[batch_size, input_tokens]`. Here the batch contains one request. The attention mask marks real input tokens with `1`; padding tokens would have `0`. Formatting, tokenization, and input transfer happen outside the generation timer.

In [5]:
messages = [{"role": "user", "content": PROMPT}]  # Represent the prompt as a user message.
rendered_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)  # Add message boundaries and the assistant prefix.
print("Formatted prompt:\n", rendered_prompt)  # Inspect the exact text passed to the tokenizer.

# The chat template already includes special tokens; do not add them a second time.
inputs = tokenizer(rendered_prompt, add_special_tokens=False, return_tensors="pt").to(device)  # Tokenize and move tensors to the model's device.
synchronize()  # Finish input transfers before any generation measurement.
input_tokens = inputs["input_ids"].shape[1]  # Read sequence length from [batch_size, sequence_length].
print("Input shape:", tuple(inputs["input_ids"].shape))  # Confirm there is one request in the batch.
print("Input device:", inputs["input_ids"].device)  # Confirm inputs and model use the same device.
print("Input tokens:", input_tokens)  # Include template tokens in the workload count.
print("First 20 token IDs:", inputs["input_ids"][0, :20].tolist())  # Inspect the numeric representation of text.
print("Attention mask:", inputs["attention_mask"][0].tolist())  # Inspect which input positions are real tokens.

Formatted prompt:
 <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain the difference between a CPU and a GPU in three short sentences.<|im_end|>
<|im_start|>assistant

Input shape: (1, 44)
Input device: cpu
Input tokens: 44
First 20 token IDs: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 5. Generate one response

`generate()` repeatedly predicts the next token until an end-of-sequence token or the output limit is reached. Greedy decoding chooses the highest-scoring token. `use_cache=True` reuses attention keys and values within this request; it does not cache an answer across requests.

`torch.inference_mode()` disables gradient bookkeeping. The returned sequence includes the input, so we slice it off before counting and decoding the response.

Our timer covers **in-process generation**, including prefill, decoding, and generation-loop overhead. It excludes loading, tokenization, input transfer, decoding IDs to text, and printing. GPU synchronization surrounds the measurement. This is neither API latency nor decode-only speed.

In [6]:
generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,  # Limit newly generated tokens without counting the prompt.
    do_sample=False,  # Choose the highest-scoring next token instead of sampling.
    num_beams=1,  # Generate a single greedy continuation without beam search.
    use_cache=True,  # Reuse attention keys and values within this request.
    eos_token_id=model.generation_config.eos_token_id,  # Stop at the model's end-of-sequence token(s).
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,  # Supply a valid padding token ID.
)

def generate_once():
    # Return both the generated response and measurements from one independent request.
    synchronize()  # Exclude previously queued device work from this request's timer.
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)  # Reset the peak counter, retaining the loaded model allocation.
    start = time.perf_counter()  # Start timing immediately before generation.
    with torch.inference_mode():  # Disable gradient bookkeeping during inference.
        output = model.generate(**inputs, generation_config=generation_config)  # Generate tokens using the input IDs and attention mask.
    synchronize()  # Wait for generation to finish before stopping the timer.
    seconds = time.perf_counter() - start  # Measure generation wall time, including synchronization.
    generated_ids = output[0, input_tokens:].cpu().tolist()  # Remove prompt tokens and copy response IDs to a Python list.
    return {
        "input_tokens": input_tokens,  # Record the prompt's actual token length.
        "output_tokens": len(generated_ids),  # Count actual generated IDs, including any terminal token.
        "generation_seconds": seconds,  # Store unrounded generation latency.
        "output_tokens_per_second": len(generated_ids) / seconds,  # Divide generated tokens by generation time.
        "cuda_peak_allocated_gib": torch.cuda.max_memory_allocated(device) / 2**30 if device.type == "cuda" else None,  # Record CUDA peak allocation; leave unavailable metrics null.
        "generated_token_ids": generated_ids,  # Preserve exact output IDs in the JSON results.
        "response": tokenizer.decode(generated_ids, skip_special_tokens=True),  # Preserve readable output beside its measurements.
    }

first_request = generate_once()  # Use the first request as a separately recorded warmup.
print(first_request["response"])  # Read the actual model response.
print(f"First request: {first_request['output_tokens']} output tokens in {first_request['generation_seconds']:.3f}s")  # Inspect warmup latency.

A CPU (Central Processing Unit) is responsible for executing instructions and performing arithmetic and logical operations, while a GPU (Graphics Processing Unit) is designed to handle parallel processing tasks, such as rendering graphics and performing complex calculations in parallel.
First request: 47 output tokens in 15.116s


## 6. Warmed-up baseline

The first request above is our warmup and is excluded from the summary. Each repetition starts a new request with a new KV cache. Reusing the same prompt controls the workload, not a response cache.

Count actual generated token IDs, including a terminal special token if produced. A token cap is not an exact output length. CUDA peak allocated memory includes the loaded model and request allocations; it is not total GPU usage. CPU/MPS peak memory is left unavailable (`null`), rather than reported as zero.

Three repetitions provide a first observation, not a reliable p95 estimate. Do not expect an exact speedup or identical generated text across different hardware.

In [7]:
measurements = []  # Keep every measured response and its metrics, rather than just averages.
for repeat in range(1, MEASURED_RUNS + 1):  # Run the same workload the configured number of times.
    row = {"repeat": repeat, **generate_once()}  # Label this request and retain its full result.
    measurements.append(row)  # Preserve the response, token IDs, latency, throughput, and memory.
    print(f"Run {repeat}: {row['output_tokens']} tokens | {row['generation_seconds']:.3f}s | {row['output_tokens_per_second']:.2f} tokens/s")  # Show this request's measurements.
    print("Response:", row["response"], "\n")  # Inspect each response next to its measurements.

summary = {
    "measured_requests": len(measurements),  # Report the sample count, excluding the warmup.
    "median_generation_seconds": statistics.median(row["generation_seconds"] for row in measurements),  # Summarize warmed-up request latency.
    "median_output_tokens_per_second": statistics.median(row["output_tokens_per_second"] for row in measurements),  # Summarize per-request output rate.
}
print(json.dumps(summary, indent=2))  # Display the summary without discarding individual results.

Run 1: 47 tokens | 15.053s | 3.12 tokens/s
Response: A CPU (Central Processing Unit) is responsible for executing instructions and performing arithmetic and logical operations, while a GPU (Graphics Processing Unit) is designed to handle parallel processing tasks, such as rendering graphics and performing complex calculations in parallel. 

Run 2: 47 tokens | 15.074s | 3.12 tokens/s
Response: A CPU (Central Processing Unit) is responsible for executing instructions and performing arithmetic and logical operations, while a GPU (Graphics Processing Unit) is designed to handle parallel processing tasks, such as rendering graphics and performing complex calculations in parallel. 

Run 3: 47 tokens | 15.396s | 3.05 tokens/s
Response: A CPU (Central Processing Unit) is responsible for executing instructions and performing arithmetic and logical operations, while a GPU (Graphics Processing Unit) is designed to handle parallel processing tasks, such as rendering graphics and performing complex

## 7. Save this run before switching runtimes

Each benchmark produces two files with the same unique filename stem under `runs/first-inference/`, relative to the notebook's working directory:

- **JSON:** environment, prompt/template, model revision, generation settings, warmup response, every measured response and its token IDs/metrics, and summary.
- **CSV:** one row per request, including the warmup, with its phase, repeat number, prompt, full response text, token counts, latency, output rate, and CUDA peak memory. Open it in a table viewer to inspect responses beside their measurements. Multiline responses are quoted as a single CSV field.

The warmup has phase `warmup` and repeat `0`; benchmark repetitions have phase `measured` and repeats `1` onward. Only measured requests contribute to the summary. No weights are saved here. This directory is ignored by Git.

Run the save cell after benchmarking. In Colab, download both files using the optional cell below **before switching runtime**. Runtime storage is temporary. Keep the CPU files for the GPU comparison. Selected reviewed results can later be copied into the repository's `results/` directory.

In [8]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]  # Give this benchmark a unique timestamped identity.
result = {
    "benchmark_version": "first-inference-v1",  # Identify the measurement procedure.
    "run_id": run_id,  # Link JSON and CSV files to the same benchmark.
    "environment": environment,  # Preserve the hardware and software context.
    "model": MODEL_NAME,  # Record which pretrained model ran.
    "resolved_revision": resolved_revision,  # Pin the model/tokenizer version for comparison.
    "attention_implementation": model.config._attn_implementation,  # Record the attention implementation actually configured.
    "messages": messages,  # Preserve the original user prompt and role.
    "rendered_prompt": rendered_prompt,  # Preserve the complete chat-formatted input.
    "input_token_ids": inputs["input_ids"][0].cpu().tolist(),  # Save the exact tokenized workload.
    "generation_config": generation_config.to_dict(),  # Save all generation settings.
    "seed": 42,  # Record the seed used during setup.
    "warmup_requests": 1,  # State how many requests were excluded from the summary.
    "download_and_load_seconds": download_and_load_seconds,  # Keep setup time separate from request latency.
    "timing_scope": "model.generate plus device synchronization; excludes tokenization, input transfer and text decoding",  # Define the generation timer.
    "first_request": first_request,  # Retain the warmup response, token IDs, and measurements.
    "measurements": measurements,  # Retain every measured response, token IDs, and measurements.
    "summary": summary,  # Include the warmed-up medians for quick comparison.
}
output_dir = Path("runs") / "first-inference"  # Keep generated artifacts in an ignored output directory.
output_dir.mkdir(parents=True, exist_ok=True)  # Create the output directory if needed.
result_path = output_dir / f"{device.type}-{run_id}.json"  # Separate CPU/GPU runs with unique filenames.
result_path.write_text(json.dumps(result, indent=2) + "\n")  # Save full results without rounding the measurements.

csv_path = result_path.with_suffix(".csv")  # Pair the readable response table with the JSON file.
fieldnames = ["phase", "repeat", "prompt", "response", "input_tokens", "output_tokens", "generation_seconds", "output_tokens_per_second", "cuda_peak_allocated_gib"]  # Define one row's columns.
requests = [{"phase": "warmup", "repeat": 0, **first_request}] + [{"phase": "measured", **row} for row in measurements]  # Include every response while labeling the warmup.
with csv_path.open("w", newline="", encoding="utf-8") as file:  # Let the CSV writer handle multiline text and quoting.
    writer = csv.DictWriter(file, fieldnames=fieldnames, extrasaction="ignore")  # Keep token-ID arrays in JSON, rather than duplicating them in CSV.
    writer.writeheader()  # Name the columns for table viewers.
    for request in requests:  # Write the warmup and each measured request independently.
        writer.writerow({"prompt": PROMPT, **request})  # Store each full response beside its own measurements.
print("Saved JSON:", result_path.resolve())  # Locate the complete benchmark artifact.
print("Saved CSV:", csv_path.resolve())  # Locate the per-request response table.

Saved JSON: /content/runs/first-inference/cpu-20261122T055909Z-87011252.json
Saved CSV: /content/runs/first-inference/cpu-20261122T055909Z-87011252.csv


In [9]:
# Optional, Colab only: uncomment these lines to download both benchmark files.
from google.colab import files  # Access Colab's browser download helper.
files.download(str(result_path))  # Download the complete JSON results.
files.download(str(csv_path))  # Download the CSV containing every response and its measurements.

## 8. Repeat on GPU and interpret the result

1. Save the CPU JSON and CSV, and copy its `resolved_revision` into `MODEL_REVISION` above.
2. In Colab, change the runtime hardware accelerator to a GPU. The runtime may restart.
3. Keep `FORCE_CPU = False` and run all cells in order. Confirm the selected device is `cuda:0` and inspect the GPU name. Without an accelerator, the notebook will still use CPU.
4. Download the GPU JSON and CSV. Compare median generation latency and median output tokens/second alongside actual output lengths and responses.
5. Calculate the **observed latency speedup** as `CPU median generation seconds / GPU median generation seconds`. If token counts differ, that ratio also reflects the changed amount of generated work. Compare output tokens/second too and report the difference.

Keep FP32 for this first comparison. CPU and GPU in separate Colab sessions also differ in host resources; this is a comparison of the recorded environments. Reduced precision, fixed exact output lengths, multiple prompts, and broader benchmarking follow in later experiments.

Record your answers after running:

- Selected CPU/GPU hardware and model revision:
- Input tokens and actual generated tokens:
- First-request time versus warmed-up median:
- CPU versus GPU latency and output rate:
- Did output text or token count change?
- What is excluded from the generation timer?
- Why are `eval()` and `inference_mode()` both used?
- What question should the next experiment investigate?

**Sources:** [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Transformers loading options](https://huggingface.co/docs/transformers/v4.57.1/main_classes/model), [PyTorch CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html), [PyTorch MPS backend](https://docs.pytorch.org/docs/stable/notes/mps.html).